# 02 – Nettoyage de HVFHV

Nettoyage de la table HVFHV (Uber, Lyft…) pour l'objectif 6 : segmentation des zones de New York. Même méthode que Yellow, Green et FHV : on **supprime** une course seulement si elle ne peut pas être rattachée à une zone ou à la période ; sinon on **vide** uniquement la mesure impossible pour garder la course dans l'activité de sa zone.

## Cellule 0 : chargement de HVFHV

On lit le fichier `hvfhv_renomme.csv` par blocs de 2 millions de lignes pour ne pas saturer la mémoire. Chaque colonne reçoit un type compact (nombres en `float32`, indicateurs en `category`) et les 3 colonnes de dates sont converties en vraies dates.

**Effet sur les données :** toutes les courses HVFHV sont en mémoire, prêtes à être nettoyées.

In [2]:
import os, time, shutil, gc, warnings
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings("ignore")

RENOM_DIR = os.path.join(os.getcwd(), "renamed_data")
CLEAN_DIR = os.path.join(os.getcwd(), "clean_data")
os.makedirs(CLEAN_DIR, exist_ok=True)

TYPES = {"zone_depart": "float32", "zone_arrivee": "float32", "distance": "float32", "prix": "float32",
         "pourboire": "float32", "frais_cbd": "float32", "paie_chauffeur": "float32",
         "partage": "category", "acces_pmr": "category"}
DATES = ["date_demande", "date_debut", "date_fin"]

t0 = time.time()
morceaux = []
for bloc in pd.read_csv(os.path.join(RENOM_DIR, "hvfhv_renomme.csv"), dtype=TYPES, chunksize=2_000_000):
    for c in DATES:
        bloc[c] = pd.to_datetime(bloc[c], format="ISO8601", errors="coerce")
    morceaux.append(bloc)

hvfhv = pd.concat(morceaux, ignore_index=True)
del morceaux
gc.collect()
hvfhv["partage"] = hvfhv["partage"].astype("category")
hvfhv["acces_pmr"] = hvfhv["acces_pmr"].astype("category")

N_DEPART = len(hvfhv)
print("Lignes :", N_DEPART)
print("Mémoire (Go) :", round(hvfhv.memory_usage(deep=True).sum() / 1e9, 2))
print("Temps (s) :", round(time.time() - t0))
display(hvfhv.head())

Lignes : 4048905
Mémoire (Go) : 0.22
Temps (s) : 20


,date_demande,date_debut,date_fin,zone_depart,zone_arrivee,distance,prix,pourboire,frais_cbd,partage,acces_pmr,paie_chauffeur
0,2024-02-01 00:05:14,2024-02-01 00:08:56,2024-02-01 00:26:03,179.0,226.0,2.520,15.180000,0.0,NaN,N,N,12.970000
1,2024-02-01 00:46:46,2024-02-01 00:48:52,2024-02-01 01:04:40,211.0,233.0,2.901,28.360001,0.0,NaN,N,N,21.129999
2,2024-02-01 00:14:30,2024-02-01 00:15:42,2024-02-01 00:27:40,74.0,167.0,3.400,17.000000,0.0,NaN,N,N,11.220000
3,2024-02-01 00:44:09,2024-02-01 00:47:13,2024-02-01 01:30:21,164.0,189.0,10.490,52.650002,0.0,NaN,N,N,38.110001
4,2024-02-01 00:08:48,2024-02-01 00:11:39,2024-02-01 00:16:26,14.0,14.0,1.500,8.890000,1.0,NaN,N,N,5.390000


## Cellule 1 : description générale

Avant de toucher aux données, on fait un diagnostic complet : `info()`, puis un résumé par colonne (type, valeurs remplies, valeurs vides, pourcentage de vides, nombre de valeurs distinctes). On compte aussi les doublons, on affiche les statistiques des colonnes numériques avec les centiles 1 % et 99 % pour repérer les valeurs extrêmes, les valeurs des indicateurs Y/N et la période couverte par chaque date.

Le nombre de valeurs distinctes n'est pas calculé sur les dates, car ce serait trop long sur des dizaines de millions de lignes.

**Effet sur les données :** aucun, c'est le point de départ qui justifie chaque étape du nettoyage.

In [3]:
hvfhv.info(memory_usage="deep", show_counts=True)

colonnes_sans_dates = [c for c in hvfhv.columns if c not in DATES]

resume = pd.DataFrame()
resume["type"] = hvfhv.dtypes.astype(str)
resume["nb_non_null"] = hvfhv.notna().sum()
resume["nb_null"] = hvfhv.isna().sum()
resume["pct_null"] = (hvfhv.isna().mean() * 100).round(3)
resume["nb_distinct"] = hvfhv[colonnes_sans_dates].nunique()
resume["exemple"] = hvfhv.iloc[0]
display(resume)

nb_doublons = hvfhv.duplicated().sum()
print("Doublons :", nb_doublons, "(", round(nb_doublons / len(hvfhv) * 100, 3), "% )")
print("Lignes avec au moins un vide :", hvfhv.isna().any(axis=1).sum())

display(hvfhv.describe(include="number", percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2))

print("partage :", hvfhv["partage"].value_counts(dropna=False).to_dict())
print("acces_pmr :", hvfhv["acces_pmr"].value_counts(dropna=False).to_dict())

for c in DATES:
    print(c, ": du", hvfhv[c].min(), "au", hvfhv[c].max())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4048905 entries, 0 to 4048904
Data columns (total 12 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   date_demande    4048905 non-null  datetime64[ns]
 1   date_debut      4048905 non-null  datetime64[ns]
 2   date_fin        4048905 non-null  datetime64[ns]
 3   zone_depart     4048905 non-null  float32       
 4   zone_arrivee    4048905 non-null  float32       
 5   distance        4048905 non-null  float32       
 6   prix            4048905 non-null  float32       
 7   pourboire       4048905 non-null  float32       
 8   frais_cbd       2007024 non-null  float32       
 9   partage         4048905 non-null  category      
 10  acces_pmr       4048905 non-null  category      
 11  paie_chauffeur  4048905 non-null  float32       
dtypes: category(2), datetime64[ns](3), float32(7)
memory usage: 208.5 MB


,type,nb_non_null,nb_null,pct_null,nb_distinct,exemple
date_demande,datetime64[ns],4048905,0,0.00,NaN,2024-02-01 00:05:14
date_debut,datetime64[ns],4048905,0,0.00,NaN,2024-02-01 00:08:56
date_fin,datetime64[ns],4048905,0,0.00,NaN,2024-02-01 00:26:03
zone_depart,float32,4048905,0,0.00,260.0,179.0
zone_arrivee,float32,4048905,0,0.00,261.0,226.0
distance,float32,4048905,0,0.00,37002.0,2.52
prix,float32,4048905,0,0.00,23042.0,15.18
pourboire,float32,4048905,0,0.00,4782.0,0.0
frais_cbd,float32,2007024,2041881,50.43,2.0,NaN
partage,category,4048905,0,0.00,2.0,N


Doublons : 0 ( 0.0 % )
Lignes avec au moins un vide : 2041881


,count,mean,std,min,1%,25%,50%,75%,99%,max
zone_depart,4048905.0,138.39,74.81,2.00,7.00,75.00,139.00,210.00,261.00,265.00
zone_arrivee,4048905.0,142.01,77.65,1.00,4.00,76.00,141.00,216.00,265.00,265.00
distance,4048905.0,4.96,5.76,0.00,0.47,1.52,2.92,6.18,26.85,317.02
prix,4048905.0,25.64,21.77,-77.60,6.60,12.50,19.35,30.75,111.56,1140.57
pourboire,4048905.0,1.09,3.27,0.00,0.00,0.00,0.00,0.00,15.58,208.97
frais_cbd,2007024.0,0.52,0.71,0.00,0.00,0.00,0.00,1.50,1.50,1.50
paie_chauffeur,4048905.0,19.56,16.59,-23.34,4.11,8.71,14.64,24.63,80.63,889.98


partage : {'N': 3906674, 'Y': 142231}
acces_pmr : {'N': 4038573, 'Y': 10332}
date_demande : du 2024-01-31 23:47:49 au 2025-03-31 23:57:57
date_debut : du 2024-02-01 00:00:02 au 2025-03-31 23:59:52
date_fin : du 2024-02-01 00:02:58 au 2025-04-01 01:10:38


## Cellule 2 : suppression des doublons

Un doublon est une course enregistrée deux fois avec exactement les mêmes 12 valeurs. On garde la première occurrence et on supprime les copies, pour qu'une même course ne soit pas comptée deux fois dans l'activité d'une zone.

**Effet sur les données :** le nombre de lignes baisse du nombre de doublons.

In [6]:
avant = len(hvfhv)
masque = hvfhv.duplicated(keep="first")
print("Doublons trouvés :", masque.sum())

if masque.sum() > 0:
    display(hvfhv[hvfhv.duplicated(keep=False)].sort_values(["date_debut", "zone_depart"]).head(6))

hvfhv = hvfhv[~masque].reset_index(drop=True)



Doublons trouvés : 0


## Cellule 3 : valeurs vides

**frais_cbd** : le péage de congestion du centre de Manhattan n'existe que depuis le 5 janvier 2025. On vérifie d'abord que les vides correspondent bien à 2024, puis on les remplace par 0, car un vide veut dire « pas de péage ».

**Dates de début et de fin** : une course sans l'une de ces dates ne peut être ni placée dans la période ni mesurée en durée, donc on la supprime.

**Effet sur les données :** plus aucun vide dans `frais_cbd`, et les rares courses sans date disparaissent.

In [7]:
annee = hvfhv["date_debut"].dt.year
verif = pd.DataFrame()
verif["nb_courses"] = annee.value_counts()
verif["nb_null_cbd"] = hvfhv["frais_cbd"].isna().groupby(annee).sum()
verif = verif.fillna(0).astype(int).sort_index()
verif["pct_null_cbd"] = (verif["nb_null_cbd"] / verif["nb_courses"] * 100).round(2)
display(verif)

print("Vides frais_cbd remplacés par 0 :", hvfhv["frais_cbd"].isna().sum())
hvfhv["frais_cbd"] = hvfhv["frais_cbd"].fillna(0)

avant = len(hvfhv)
hvfhv = hvfhv.dropna(subset=["date_debut", "date_fin"]).reset_index(drop=True)
print("Courses sans date supprimées :", avant - len(hvfhv))

display(hvfhv.isna().sum().to_frame("nb_null"))

,nb_courses,nb_null_cbd,pct_null_cbd
date_debut,,,
2024,2041881,2041881,100.0
2025,2007024,0,0.0


Vides frais_cbd remplacés par 0 : 2041881
Courses sans date supprimées : 0


,nb_null
date_demande,0
date_debut,0
date_fin,0
zone_depart,0
zone_arrivee,0
distance,0
prix,0
pourboire,0
frais_cbd,0
partage,0


## Cellule 4 : filtre sur la période

L'étude porte sur février et mars 2024 comparés à février et mars 2025, soit 119 jours. Les courses datées en dehors de ces 4 mois sont des erreurs d'horloge ou des déclarations tardives : elles fausseraient le nombre de courses par jour et la croissance 2024 → 2025. On garde uniquement les courses dont la date de début tombe dans ces 4 mois.

**Effet sur les données :** quelques lignes en moins, et la période devient exactement celle de l'étude.

In [8]:
MOIS_ETUDE = ["2024-02", "2024-03", "2025-02", "2025-03"]

avant = len(hvfhv)
mois = hvfhv["date_debut"].dt.strftime("%Y-%m")
garder = mois.isin(MOIS_ETUDE)

print("Courses hors période :")
display(mois[~garder].value_counts().sort_index().to_frame("nb_courses"))

hvfhv = hvfhv[garder].reset_index(drop=True)
del mois, garder
gc.collect()
print("Avant :", avant, "-> après :", len(hvfhv), "| supprimées :", avant - len(hvfhv))
print("Période : du", hvfhv["date_debut"].min(), "au", hvfhv["date_debut"].max())
display(hvfhv["date_debut"].dt.strftime("%Y-%m").value_counts().sort_index().to_frame("nb_courses"))

Courses hors période :


,nb_courses
date_debut,


Avant : 4048905 -> après : 4048905 | supprimées : 0
Période : du 2024-02-01 00:00:02 au 2025-03-31 23:59:52


,nb_courses
date_debut,
2024-02,972048
2024-03,1069833
2025-02,974112
2025-03,1032912


## Cellule 5 : durée de la course

On calcule `duree_min` = date de fin − date de début, en minutes. Une durée négative, inférieure à 1 minute ou supérieure à 3 heures est impossible pour une course en ville (erreur d'horodatage, course annulée).

On **vide** la durée invalide sans supprimer la course : la course a bien eu lieu dans sa zone et compte dans l'activité, mais elle ne fausse pas la durée moyenne.

**Effet sur les données :** une colonne de plus, nombre de lignes inchangé.

In [9]:
DUREE_MIN = 1
DUREE_MAX = 180

hvfhv["duree_min"] = ((hvfhv["date_fin"] - hvfhv["date_debut"]).dt.total_seconds() / 60).astype("float32")
display(hvfhv["duree_min"].describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]).to_frame().T.round(2))

d = hvfhv["duree_min"]
print("Durée négative :", (d < 0).sum())
print("Durée < 1 min :", ((d >= 0) & (d < DUREE_MIN)).sum())
print("Durée > 180 min :", (d > DUREE_MAX).sum())

invalide = (d < DUREE_MIN) | (d > DUREE_MAX)
hvfhv.loc[invalide, "duree_min"] = np.nan
print("Durées vidées :", invalide.sum(), "| lignes :", len(hvfhv))

,count,mean,std,min,0.1%,1%,50%,99%,99.9%,max
duree_min,4048905.0,19.26,13.52,0.0,1.9,3.18,15.77,66.83,98.75,585.08


Durée négative : 0
Durée < 1 min : 309
Durée > 180 min : 131
Durées vidées : 440 | lignes : 4048905


## Cellule 6 : temps d'attente (propre à HVFHV)

On calcule `attente_min` = prise en charge − commande dans l'application, en minutes. Cette mesure dit à quelle vitesse un VTC arrive dans chaque zone, c'est un bon indicateur de l'offre disponible. Une attente négative ou supérieure à 60 minutes est une erreur : on la vide, la course reste.

**Effet sur les données :** une colonne de plus, nombre de lignes inchangé.

In [10]:
ATTENTE_MAX = 60

hvfhv["attente_min"] = ((hvfhv["date_debut"] - hvfhv["date_demande"]).dt.total_seconds() / 60).astype("float32")
display(hvfhv["attente_min"].describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]).to_frame().T.round(2))

a = hvfhv["attente_min"]
print("Date de demande vide :", a.isna().sum())
print("Attente négative :", (a < 0).sum())
print("Attente > 60 min :", (a > ATTENTE_MAX).sum())

invalide = (a < 0) | (a > ATTENTE_MAX)
hvfhv.loc[invalide, "attente_min"] = np.nan
print("Attentes vidées :", invalide.sum(), "| lignes :", len(hvfhv))

,count,mean,std,min,0.1%,1%,50%,99%,99.9%,max
attente_min,4048905.0,4.51,3.34,-54.35,-18.28,0.27,3.93,15.28,24.92,131.12


Date de demande vide : 0
Attente négative : 39701
Attente > 60 min : 360
Attentes vidées : 40061 | lignes : 4048905


## Cellule 7 : contrôle des zones

D'après le fichier `taxi_zone_lookup.csv`, les zones 1 à 263 sont de vraies zones de New York ; 264 veut dire « inconnue » et 265 « hors de NYC ».

- **Zone de départ** invalide : la course ne peut être rattachée à aucune zone, on la **supprime**.
- **Zone d'arrivée** invalide : la course a bien commencé dans une vraie zone, on la **garde** et on vide seulement l'arrivée.

**Effet sur les données :** quelques lignes en moins, toutes les zones de départ restantes sont réelles.

In [11]:
avant = len(hvfhv)
depart_ok = hvfhv["zone_depart"].between(1, 263)
arrivee_ok = hvfhv["zone_arrivee"].between(1, 263)

print("Zones de départ invalides :")
display(hvfhv.loc[~depart_ok, "zone_depart"].value_counts(dropna=False).to_frame("nb_courses"))
print("Zones d'arrivée invalides :")
display(hvfhv.loc[~arrivee_ok, "zone_arrivee"].value_counts(dropna=False).to_frame("nb_courses"))

hvfhv.loc[~arrivee_ok, "zone_arrivee"] = np.nan
hvfhv = hvfhv[depart_ok].reset_index(drop=True)
del depart_ok, arrivee_ok
gc.collect()

hvfhv["zone_depart"] = hvfhv["zone_depart"].astype("int16")
hvfhv["zone_arrivee"] = hvfhv["zone_arrivee"].astype("Int16")
print("Zones de départ distinctes :", hvfhv["zone_depart"].nunique())

Zones de départ invalides :


,nb_courses
zone_depart,
265.0,161


Zones d'arrivée invalides :


,nb_courses
zone_arrivee,
265.0,166165


Zones de départ distinctes : 259


## Cellule 8 : distance

Une distance nulle vient d'une course annulée ou d'un GPS qui n'a rien enregistré ; une distance de plusieurs centaines de miles est une erreur (NYC fait environ 35 miles de large). On garde l'intervalle ]0 ; 100] miles et on **vide** le reste sans supprimer la course.

**Effet sur les données :** la distance moyenne d'une zone n'est plus faussée, nombre de lignes inchangé.

In [12]:
DIST_MAX = 100

display(hvfhv["distance"].describe(percentiles=[0.01, 0.5, 0.99, 0.999]).to_frame().T.round(2))

d = hvfhv["distance"]
print("Distance négative :", (d < 0).sum())
print("Distance = 0 :", (d == 0).sum())
print("Distance > 100 :", (d > DIST_MAX).sum())

invalide = (d <= 0) | (d > DIST_MAX)
hvfhv.loc[invalide, "distance"] = np.nan
print("Distances vidées :", invalide.sum(), "| lignes :", len(hvfhv))

,count,mean,std,min,1%,50%,99%,99.9%,max
distance,4048744.0,4.96,5.76,0.0,0.47,2.92,26.85,51.98,317.02


Distance négative : 0
Distance = 0 : 543
Distance > 100 : 252
Distances vidées : 795 | lignes : 4048744


## Cellule 9 : prix

Un prix négatif ou nul correspond à une annulation ou un remboursement ; un prix de plusieurs milliers de dollars est une erreur de saisie. On garde l'intervalle ]0 ; 500] $ et on **vide** le reste sans supprimer la course.

**Effet sur les données :** le prix moyen et le prix par mile d'une zone ne sont plus faussés, nombre de lignes inchangé.

In [13]:
PRIX_MAX = 500

display(hvfhv["prix"].describe(percentiles=[0.01, 0.5, 0.99, 0.999]).to_frame().T.round(2))

p = hvfhv["prix"]
print("Prix négatif :", (p < 0).sum())
print("Prix = 0 :", (p == 0).sum())
print("Prix > 500 :", (p > PRIX_MAX).sum())

invalide = (p <= 0) | (p > PRIX_MAX)
hvfhv.loc[invalide, "prix"] = np.nan
print("Prix vidés :", invalide.sum(), "| lignes :", len(hvfhv))

,count,mean,std,min,1%,50%,99%,99.9%,max
prix,4048744.0,25.64,21.77,-77.6,6.6,19.35,111.56,205.12,1140.57


Prix négatif : 523
Prix = 0 : 3355
Prix > 500 : 66
Prix vidés : 3944 | lignes : 4048744


## Cellule 10 : pourboire

Un pourboire négatif est une annulation, et un pourboire supérieur à 200 $ est presque toujours une erreur. On **vide** ces valeurs sans supprimer la course.

Contrairement aux taxis, tout est payé dans l'application, donc le pourboire est toujours enregistré : il n'y a pas de pourboires en espèces invisibles.

**Effet sur les données :** le pourboire moyen d'une zone n'est plus faussé, nombre de lignes inchangé.

In [15]:
PB_MAX = 200

display(hvfhv["pourboire"].describe(percentiles=[0.01, 0.5, 0.99, 0.999]).to_frame().T.round(2))

pb = hvfhv["pourboire"]
print("Pourboire négatif :", (pb < 0).sum())
print("Pourboire > 200 :", (pb > PB_MAX).sum())
print("Part des courses avec pourboire (%) :", round((pb > 0).mean() * 100, 1))

invalide = (pb < 0) | (pb > PB_MAX)
hvfhv.loc[invalide, "pourboire"] = np.nan

,count,mean,std,min,1%,50%,99%,99.9%,max
pourboire,4048743.0,1.09,3.27,0.0,0.0,0.0,15.58,31.94,135.18


Pourboire négatif : 0
Pourboire > 200 : 0
Part des courses avec pourboire (%) : 18.4


## Cellule 11 : paie du chauffeur (propre à HVFHV)

`paie_chauffeur` est ce que le chauffeur reçoit, hors pourboire, péages et taxes. Comparée au prix, elle donne la part qui revient au chauffeur dans chaque zone. Une paie négative, nulle ou supérieure à 500 $ est une erreur : on la **vide**, la course reste.

**Effet sur les données :** nombre de lignes inchangé.

In [16]:
PAIE_MAX = 500

display(hvfhv["paie_chauffeur"].describe(percentiles=[0.01, 0.5, 0.99, 0.999]).to_frame().T.round(2))

pc = hvfhv["paie_chauffeur"]
print("Paie <= 0 :", (pc <= 0).sum())
print("Paie > 500 :", (pc > PAIE_MAX).sum())

invalide = (pc <= 0) | (pc > PAIE_MAX)
hvfhv.loc[invalide, "paie_chauffeur"] = np.nan
print("Paies vidées :", invalide.sum(), "| lignes :", len(hvfhv))

,count,mean,std,min,1%,50%,99%,99.9%,max
paie_chauffeur,4048744.0,19.56,16.59,-23.34,4.11,14.64,80.63,144.37,889.98


Paie <= 0 : 962
Paie > 500 : 18
Paies vidées : 980 | lignes : 4048744


## Cellule 12 : péage CBD

Pour les VTC, le péage du centre de Manhattan vaut **1,50 $** par course (le double des taxis, 0,75 $).

- Un montant **négatif** est une annulation du péage : on le ramène à 0.
- Un montant **positif différent de 1,50 $** est une erreur de saisie sur le montant, mais il prouve que la course a traversé la zone CBD : on le ramène à 1,50 $.

**Effet sur les données :** il ne reste que deux valeurs claires, 0 (pas de péage) ou 1,50 $ (péage payé).

In [18]:
TARIF_CBD = 1.50
annee = hvfhv["date_debut"].dt.year

display(pd.crosstab(hvfhv["frais_cbd"], annee))

negatif = hvfhv["frais_cbd"] < 0
hvfhv.loc[negatif, "frais_cbd"] = 0

inhabituel = (hvfhv["frais_cbd"] > 0) & (hvfhv["frais_cbd"] != TARIF_CBD)
hvfhv.loc[inhabituel, "frais_cbd"] = TARIF_CBD

print("Négatifs ramenés à 0 :", negatif.sum())
print("Inhabituels ramenés à 1.50 :", inhabituel.sum())


date_debut,2024,2025
frais_cbd,,
0.0,2041789,1308947
1.5,0,698008


Négatifs ramenés à 0 : 0
Inhabituels ramenés à 1.50 : 0


## Cellule 13 : indicateurs partage et acces_pmr (propres à HVFHV)

- `partage` : Y si le passager a demandé une course partagée, N sinon.
- `acces_pmr` : Y si le passager a demandé un véhicule accessible aux fauteuils roulants, N sinon.

Toute autre valeur que Y ou N est une erreur : on la vide.

**Effet sur les données :** les deux indicateurs ne contiennent plus que Y, N ou vide, nombre de lignes inchangé.

In [19]:
for c in ["partage", "acces_pmr"]:
    print(c, "AVANT :", hvfhv[c].value_counts(dropna=False).to_dict())
    valide = hvfhv[c].isin(["Y", "N"])
    hvfhv[c] = hvfhv[c].astype("object").where(valide).astype("category")
    print(c, "APRÈS :", hvfhv[c].value_counts(dropna=False).to_dict())

partage AVANT : {'N': 3906518, 'Y': 142226}
partage APRÈS : {'N': 3906518, 'Y': 142226}
acces_pmr AVANT : {'N': 4038412, 'Y': 10332}
acces_pmr APRÈS : {'N': 4038412, 'Y': 10332}


## Cellule 14 : bilan du nettoyage

On compare la table de départ à la table nettoyée : nombre de lignes gardées et supprimées, valeurs vidées par colonne, statistiques finales et période. Les vides dans `duree_min`, `attente_min`, `distance`, `prix`, `pourboire` et `paie_chauffeur` sont voulus : ce sont les mesures impossibles qu'on a vidées sans supprimer les courses.

**Effet sur les données :** aucun, c'est le contrôle final et les chiffres à mettre dans le rapport.

In [20]:
n_fin = len(hvfhv)
print("Lignes au départ :", N_DEPART)
print("Lignes gardées :", n_fin, "(", round(n_fin / N_DEPART * 100, 2), "% )")
print("Lignes supprimées :", N_DEPART - n_fin)

bilan = pd.DataFrame()
bilan["type"] = hvfhv.dtypes.astype(str)
bilan["nb_vides"] = hvfhv.isna().sum()
bilan["pct_vides"] = (hvfhv.isna().mean() * 100).round(3)
display(bilan)

display(hvfhv.describe(include="number", percentiles=[0.01, 0.5, 0.99]).T.round(2))
print("Période : du", hvfhv["date_debut"].min(), "au", hvfhv["date_debut"].max())
print("Zones de départ distinctes :", hvfhv["zone_depart"].nunique())

Lignes au départ : 4048905
Lignes gardées : 4048744 ( 100.0 % )
Lignes supprimées : 161


,type,nb_vides,pct_vides
date_demande,datetime64[ns],0,0.000
date_debut,datetime64[ns],0,0.000
date_fin,datetime64[ns],0,0.000
zone_depart,int16,0,0.000
zone_arrivee,Int16,166100,4.103
distance,float32,795,0.020
prix,float32,3944,0.097
pourboire,float32,1,0.000
frais_cbd,float32,0,0.000
partage,category,0,0.000


,count,mean,std,min,1%,50%,99%,max
zone_depart,4048744.0,138.381665,74.873496,2.0,7.0,139.0,261.0,263.0
zone_arrivee,3882644.0,136.750365,75.18581,1.0,4.0,138.0,261.0,263.0
distance,4047949.0,4.950519,5.67485,0.007,0.47,2.92,26.799999,99.879997
prix,4044800.0,25.652391,21.599541,0.01,6.72,19.370001,111.540001,499.170013
pourboire,4048743.0,1.087152,3.269491,0.0,0.0,0.0,15.58,135.179993
frais_cbd,4048744.0,0.258602,0.569641,0.0,0.0,0.0,1.5,1.5
paie_chauffeur,4047764.0,19.558329,16.539309,0.01,4.13,14.64,80.623703,492.630005
duree_min,4048304.0,19.253813,13.456769,1.0,3.2,15.766666,66.783333,178.833328
attente_min,4008685.0,4.640129,2.91026,0.0,0.95,3.95,15.283334,59.966667


Période : du 2024-02-01 00:00:02 au 2025-03-31 23:59:52
Zones de départ distinctes : 259


## Cellule 15 : sauvegarde

On enregistre la table nettoyée dans `clean_data/hvfhv_clean.csv.gz`. La compression gzip ne change aucune valeur : elle divise seulement la taille du fichier par trois environ, et pandas le relit directement avec `pd.read_csv`. On affiche l'espace libre avant d'écrire pour éviter l'erreur de disque plein.

**Effet sur les données :** aucun changement, le résultat est gardé sur le disque pour l'agrégation par zone.

In [21]:
print("Espace libre (Go) :", round(shutil.disk_usage(CLEAN_DIR).free / 1e9, 2))

t0 = time.time()
cible = os.path.join(CLEAN_DIR, "hvfhv_clean.csv.gz")
hvfhv.to_csv(cible, index=False, chunksize=1_000_000, date_format="%Y-%m-%d %H:%M:%S",
             compression={"method": "gzip", "compresslevel": 1})

print("Fichier :", cible)
print("Lignes :", len(hvfhv), "| colonnes :", hvfhv.shape[1])
print("Taille (Go) :", round(os.path.getsize(cible) / 1e9, 2), "| temps (s) :", round(time.time() - t0))
print(list(hvfhv.columns))

Espace libre (Go) : 1.71
Fichier : c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\clean_data\hvfhv_clean.csv.gz
Lignes : 4048744 | colonnes : 14
Taille (Go) : 0.14 | temps (s) : 48
['date_demande', 'date_debut', 'date_fin', 'zone_depart', 'zone_arrivee', 'distance', 'prix', 'pourboire', 'frais_cbd', 'partage', 'acces_pmr', 'paie_chauffeur', 'duree_min', 'attente_min']
